# ADAS-Vision — Milestone 3: Huấn luyện & Benchmark YOLO11 trên BDD100K

Notebook này hướng dẫn quy trình huấn luyện và đánh giá mô hình **YOLO11** (Ultralytics) cho bài toán Object Detection (9 classes) trên tập dữ liệu **BDD100K** sử dụng **Google Colab (GPU T4)**.

### Các lớp nhận diện (DET_CLASSES):
`car, truck, bus, motorcycle, bicycle, pedestrian, rider, traffic_light, traffic_sign`

## 1. Kiểm tra phần cứng GPU & Môi trường

In [ ]:
!nvidia-smi

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Kết nối Google Drive & Cài đặt Thư viện

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Clone repository hoặc chuyển vào thư mục code đã mount trên Drive
# %cd /content/drive/MyDrive/Developing-utility-vehicles-and-regional-transit-networks

!pip install --upgrade ultralytics albumentations pyyaml opencv-python-headless

## 3. Cấu hình Dữ liệu BDD100K YOLO

In [ ]:
from pathlib import Path

import yaml

# Cập nhật đường dẫn dataset trên Colab nếu cần
data_cfg = {
    'path': '/content/bdd100k_yolo',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'car',
        1: 'truck',
        2: 'bus',
        3: 'motorcycle',
        4: 'bicycle',
        5: 'pedestrian',
        6: 'rider',
        7: 'traffic_light',
        8: 'traffic_sign'
    }
}

with Path('configs/bdd_yolo.yaml').open('w') as f:
    yaml.dump(data_cfg, f, sort_keys=False)
print("Cấu hình bdd_yolo.yaml đã sẵn sàng!")

## 4. Huấn luyện Mô hình YOLO11 (yolo11s.pt / yolo11n.pt)

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình pretrained COCO
model = YOLO('yolo11s.pt')

# Chạy huấn luyện với Mixed Precision (AMP)
results = model.train(
    data='configs/bdd_yolo.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    workers=4,
    amp=True,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
    project='runs/detect',
    name='yolo11s_bdd',
    exist_ok=True
)

## 5. Đánh giá Định lượng & Benchmark Đối sánh

In [ ]:
# Đánh giá validation trên tập val BDD100K
metrics = model.val(data='configs/bdd_yolo.yaml', imgsz=640, batch=16)

print("=== BDD100K Detection Metrics ===")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

## 6. Chạy Thử nghiệm Trực quan trên Ảnh Dashcam

In [ ]:
# sample_img = 'data/bdd100k/images/100k/val/cabc30fc-e7726578.jpg'
# res = model.predict(sample_img, conf=0.35, imgsz=640)
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 7))
# plt.imshow(res[0].plot()[:, :, ::-1])
# plt.axis('off')
# plt.show()

## 7. Sao lưu Trọng số Tốt nhất (weights/detector_best.pt)

In [ ]:
import shutil

best_weights = Path('runs/detect/yolo11s_bdd/weights/best.pt')
dest = Path('weights/detector_best.pt')
dest.parent.mkdir(parents=True, exist_ok=True)

if best_weights.exists():
    shutil.copy2(best_weights, dest)
    print(f"Đã lưu thành công weights tốt nhất vào: {dest}")
else:
    print("Chưa tìm thấy best.pt, hãy kiểm tra lại đường dẫn runs/detect!")